02 - Train the study-level model
#
# Reads the mosaic corpus from notebook 01, mounted as a kernel source, and
# trains one fold. Each fold runs as its own kernel (rsna-knee-train-f0, -f1,
# ...), because pushing them all to one id would overwrite earlier weights.
#
# Validation reports two numbers: weak AUC over ~880 held-out studies (stable,
# used for checkpoint selection) and gold AUC over the handful of human-labelled
# studies in the fold (honest but very noisy).

In [ ]:
import sys, subprocess
from pathlib import Path

CODE = next(
    (p for pat in ("rsna-knee-code", "*/rsna-knee-code", "*/*/rsna-knee-code")
     for p in sorted(Path("/kaggle/input").glob(pat)) if (p / "src").is_dir()),
    None,
)
assert CODE is not None, "rsna-knee-code dataset is not attached to this notebook"
sys.path.insert(0, str(CODE / "src"))
sys.path.insert(0, str(CODE))
print("CODE", CODE)

# timm ships in the Kaggle image; the install is a fallback for images that
# lack it, and cannot run in the offline submission notebook.
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)
    import timm

import rsna_knee.config as cfg
print("RAW ", cfg.RAW, cfg.RAW.exists())
print("PREP", cfg.PREP, cfg.PREP.exists())

In [ ]:
!cd {CODE} && python scripts/build_weak_labels.py

In [ ]:
FOLD = 0
!cd {CODE} && python scripts/train.py --fold {FOLD} --epochs 8 --backbone resnet34 \
    --batch 2 --accum 8 --workers 2 --out /kaggle/working/models

In [ ]:
import json, glob
for h in sorted(glob.glob("/kaggle/working/models/*_history.json")):
    print(h)
    print(json.dumps(json.load(open(h)), indent=2))